# Phase 3 — MCNS motion pathway extraction (Colab, free T4)

Extracts the two motion **candidate** pathways from the Janelia MaleCNS v1.0 bulk files:

```
ON :  L1        -> Mi1        -> T4a / T4b / T4c / T4d
OFF:  L2        -> Tm1 / Tm2  -> T5a / T5b / T5c / T5d
```

> **Execution status: NOT YET RUN.** This notebook was written and its code paths were
> tested against tiny synthetic Feather fixtures on a local CPU. It has **not** been
> executed in Colab against the real 1.05 GB MaleCNS files. Treat the output cells as
> unverified until you run them. The first run is the experiment.

**What this does:** measures the structure of the candidate populations — cell-type counts,
edge counts, integer synapse-count distributions, and a cell-type connectivity matrix.

**What this deliberately does NOT do:** simulate anything, build a network, train a policy,
or convert synapse counts into neural weights. Those are later phases.

**`weight` is the number of synapses** from source to target body, exactly as MaleCNS
defines it. It is a structural count. It is **not** synaptic strength and is not rescaled.

**No credentials are needed.** Everything here runs on local bulk files. There is no
neuPrint call anywhere in this notebook, and the token is not read.


In [ ]:
# --- 0. Runtime ----------------------------------------------------------
# A T4 is NOT required. Extraction is memory- and I/O-bound, not compute-bound:
# the connectivity table is memory-mapped and read one record batch at a time.
import sys, time, json, platform

print("python", sys.version.split()[0], "|", platform.system())
try:
    import google.colab  # noqa: F401
    print("running in Colab")
except ImportError:
    print("not in Colab")
try:
    import psutil
    print("RAM available: %.1f GB" % (psutil.virtual_memory().available / 1e9))
except ImportError:
    print("psutil not installed; install it for memory reporting")


In [ ]:
# --- 1. Install ------------------------------------------------------------
!pip install -q -e .
!pip install -q pyarrow            # required: the MCNS files are Arrow IPC / Feather
!pip install -q pandas numpy scipy networkx matplotlib pyyaml
print("installed")


## 2. Point at your data

**Edit `MCNS_DIR` to your own directory.** Nothing in this notebook hardcodes a Drive
path. Two options:

* **Local / mounted** — a plain filesystem path. This is the default.
* **Google Drive** — uncomment the `drive.mount` and `MCNS_DIR` lines in the next cell.

Required files, with the names MaleCNS publishes:

```
body-annotations-male-cns-v1.0-minconf-0.5.feather    (~14 MB)
connectome-weights-male-cns-v1.0-minconf-0.5.feather   (~1.05 GB)
```

The 1.05 GB file is **not** in git and must not be committed.


In [ ]:
from pathlib import Path

# --- EDIT THIS to point at your own copy of the two Feather files --------
MCNS_DIR = Path("data/raw")

# Google Drive alternative (uncomment both lines):
# from google.colab import drive
# drive.mount("/content/drive")
# MCNS_DIR = Path("/content/drive/MyDrive/<your-folder>")
# ---------------------------------------------------------------------------

ANNOTATIONS = MCNS_DIR / "body-annotations-male-cns-v1.0-minconf-0.5.feather"
CONNECTIVITY = MCNS_DIR / "connectome-weights-male-cns-v1.0-minconf-0.5.feather"
OUTPUT_DIR = Path("outputs/mcns_motion")

for path in (ANNOTATIONS, CONNECTIVITY):
    print(f"{'OK ' if path.is_file() else 'MISSING'}  {path}")

if not (ANNOTATIONS.is_file() and CONNECTIVITY.is_file()):
    print("\nEdit MCNS_DIR above, or download the two files from")
    print("https://male-cns.janelia.org/download and point MCNS_DIR at them.")


## 3. Inspect the files without reading them

Metadata only. This reads the Arrow footer and **zero data rows**, so it is instant even
for the 1.05 GB table. The exact total row count comes from the pandas metadata block in
the footer, so no scan is needed to learn the file's size.


In [ ]:
import pyarrow as pa
import pyarrow.feather as feather


def footer_summary(path: Path) -> dict:
    """Schema and row count from the Arrow footer. Reads no record data."""
    source = pa.memory_map(str(path), "rb")
    reader = pa.ipc.open_file(source)
    total = -1
    meta = reader.schema.metadata or {}
    if b"pandas" in meta:
        payload = json.loads(meta[b"pandas"])
        for column in payload.get("index_columns", []) or []:
            if column.get("kind") == "range":
                total = int(column["stop"])
    return {
        "file": path.name,
        "size_gb": round(path.stat().st_size / 1e9, 3),
        "record_batches": reader.num_record_batches,
        "rows": total,
        "columns": {f.name: str(f.type) for f in reader.schema},
    }


if ANNOTATIONS.is_file():
    for path in (ANNOTATIONS, CONNECTIVITY):
        info = footer_summary(path)
        print(f"--- {info['file']}")
        print(f"    size    : {info['size_gb']} GB")
        print(f"    batches : {info['record_batches']:,}")
        print(f"    rows    : {info['rows']:,}" if info["rows"] > 0 else "    rows    : unknown")
        print(f"    columns : {info['columns']}")
else:
    print("skipped: files not present")


## 4. Run the extraction

The heavy lifting lives in `flybrain.data.motion_candidate`, which is unit-tested against
tiny fixtures and is importable on its own. The call below is the same one the CLI makes.

Two optional budgets, both **explicit** and both recorded in the report:

* `body_budget_per_type` — a deterministic, hemisphere-balanced per-cell-type body budget.
  Use `0` (the default) for the complete annotated population.
* `read_budget` — a cap on record batches read. If it binds, the report says the result is
  a weight-truncated subset. Nothing is ever truncated silently.


In [ ]:
from flybrain.data.motion_candidate import (
    MOTION_CELL_TYPES,
    MOTION_PATHWAYS,
    extract_motion_candidates,
    write_report,
)

print("ON  :", " -> ".join([MOTION_PATHWAYS["ON"]["input"], MOTION_PATHWAYS["ON"]["relay"], *MOTION_PATHWAYS["ON"]["outputs"]]))
print("OFF :", " -> ".join([MOTION_PATHWAYS["OFF"]["input"], *MOTION_PATHWAYS["OFF"]["relay"], *MOTION_PATHWAYS["OFF"]["outputs"]]))
print()
print("cell types kept separate (never collapsed into a generic T4/T5):")
print("  ", ", ".join(MOTION_CELL_TYPES))

BODY_BUDGET_PER_TYPE = 0   # 0 = complete annotated population
READ_BUDGET = 0            # 0 = read every record batch
DIRECTION = "both"        # downstream | upstream | both

started = time.perf_counter()
report = extract_motion_candidates(
    ANNOTATIONS,
    CONNECTIVITY,
    cell_types=MOTION_CELL_TYPES,
    direction=DIRECTION,
    require_superclass=True,      # official MaleCNS neuron predicate
    body_budget_per_type=BODY_BUDGET_PER_TYPE,
    read_budget=READ_BUDGET,
)
elapsed = time.perf_counter() - started
print(f"\nextraction wall time: {elapsed:.1f} s")
print("weight semantics:", report["weight_semantics"])


## 5. Cell-type counts

Bodies, hemisphere coverage, coordinate coverage, and how much connectivity each type
actually has in this graph. Out/in degree here counts only edges **within** the candidate
population, not the whole connectome.


In [ ]:
import pandas as pd

per_type = pd.DataFrame(report["populations"]["per_cell_type"]).T
per_type = per_type[["bodies", "with_somaSide", "with_coordinates", "outgoing_edges", "incoming_edges"]]
per_type.columns = ["bodies", "somaSide", "coords", "out_edges", "in_edges"]
print(f"candidate bodies: {report['populations']['candidate_bodies']:,}")
per_type


## 6. Pathway statistics and memory

The retained-edge array is the only large allocation. The dense equivalent is printed for
comparison and is **never** built.


In [ ]:
extraction = report["extraction"]
memory = report["memory"]

print(f"rows scanned            : {extraction['rows_scanned']:,}")
print(f"record batches          : {extraction['record_batches_processed']:,} of {extraction['record_batches_total']:,}")
print(f"edges retained          : {extraction['edges_retained']:,}")
print(f"extraction seconds      : {extraction['seconds']}")
print(f"total seconds           : {report['total_seconds']}")
print()
print(f"retained edge bytes     : {memory['retained_edge_bytes']/1e6:.1f} MB")
print(f"dense NxN equivalent    : {memory['dense_would_be_bytes']/1e9:.2f} GB  (never built)")
saving = memory['dense_would_be_bytes'] / max(memory['retained_edge_bytes'], 1)
print(f"memory saving           : {saving:.0f}x")
if memory["peak_process_bytes"]:
    print(f"peak process memory     : {memory['peak_process_bytes']/1e9:.2f} GB")
else:
    print("peak process memory     : unavailable on this platform")

subset = report["subset"]
if subset["is_subset"] or subset["read_truncated"]:
    print("\n*** THIS RESULT IS A SUBSET ***")
    if subset["note"]:
        print("   ", subset["note"])
    if subset["read_truncation_note"]:
        print("   ", subset["read_truncation_note"])
else:
    print("\ncomplete annotated population; no budget applied")


In [ ]:
rows = []
for name, block in report["pathways"].items():
    if not block.get("available"):
        continue
    stats = block["synapse_count_stats"]
    rows.append(
        {
            "pathway": name,
            "role": block["role"],
            "types": len(block["cell_types"]),
            "nodes": block["nodes"],
            "edges": block["edges"],
            "syn_min": stats["min"],
            "syn_median": stats["median"],
            "syn_max": stats["max"],
            "syn_total": stats["total"],
            "self_loops": block["self_loops"],
            "cross_side_edges": block["cross_side_edges"],
        }
    )
pd.DataFrame(rows)


## 7. Cell-type connectivity matrix

Total integer synapse count for every ordered cell-type pair, restricted to the candidate
population. Read the diagonal with care: those are within-type connections, and they are
usually the largest single block because a type has thousands of members.


In [ ]:
matrix = pd.DataFrame(report["cell_type_connectivity_synapses"])
print("total integer synapse count, rows = source, columns = target")
matrix


In [ ]:
import numpy as np

def show_pair(src: str, dst: str) -> None:
    if src in matrix.index and dst in matrix.columns:
        print(f"{src:>5} -> {dst:<5} : {int(matrix.loc[src, dst]):,} synapses")


print("ON channel, expected feedforward steps:")
for pair in (("L1", "Mi1"), ("Mi1", "T4a"), ("Mi1", "T4b"), ("Mi1", "T4c"), ("Mi1", "T4d")):
    show_pair(*pair)
print()
print("OFF channel, expected feedforward steps:")
for pair in (("L2", "Tm1"), ("L2", "Tm2"), ("Tm1", "T5a"), ("Tm2", "T5a")):
    show_pair(*pair)
print()
print("compare with the reverse (feedback) direction:")
for pair in (("Mi1", "L1"), ("T4a", "Mi1"), ("Tm1", "L2")):
    show_pair(*pair)


## 8. Synapse-count distributions

These are **counts of synapses**, not strengths. The distribution matters for any later
modelling decision: if it is heavily skewed towards 1-2, then a simulation that treats
every edge as equally strong will misrepresent the graph, and any conversion from count to
conductance is a modelling choice that has to be made explicitly.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

stats = report["extraction"]["synapse_count_stats"]
print("retained-edge synapse-count summary")
print(f"  count  : {stats['count']:,}")
print(f"  min    : {stats['min']}")
print(f"  median : {stats['median']}")
print(f"  mean   : {stats['mean']:.2f}")
print(f"  max    : {stats['max']}")
print(f"  total  : {stats['total']:,}")


In [ ]:
# Re-extract the retained counts to plot them. Only candidate edges are held, so this
# is cheap and touches no part of the 1.05 GB file that was not already streamed.
from flybrain.data.motion_candidate import (
    build_populations,
    extract_candidate_edges,
    load_annotations,
)

annotations = load_annotations(ANNOTATIONS)
populations = build_populations(
    annotations, MOTION_CELL_TYPES, require_superclass=True, body_budget_per_type=BODY_BUDGET_PER_TYPE
)
edges = extract_candidate_edges(CONNECTIVITY, populations, direction=DIRECTION, read_budget=READ_BUDGET)
counts = edges.synapse_count
print(f"retained edges: {counts.size:,}")

figure, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(np.clip(counts, 0, 60), bins=60, color="#2ec4b6")
axes[0].set_title("synapse count per edge (clipped at 60)")
axes[0].set_xlabel("synapses (a COUNT, not a strength)")
axes[0].set_ylabel("edges")

top = np.sort(counts)[::-1][:20000]
axes[1].hist(top, bins=50, color="#ffd166")
axes[1].set_yscale("log")
axes[1].set_title("strongest 20,000 edges")
axes[1].set_xlabel("synapses")

axes[2].hist(np.log10(counts), bins=60, color="#ef6f6c")
axes[2].set_title("log10(synapse count)")
axes[2].set_xlabel("log10 synapses")

figure.patch.set_facecolor("white")
figure.tight_layout()
plt.show()


## 9. Small graph visualisation

A **cell-type-level** graph, not a neuron-level one. 22,000 individual neurons are far
too many to lay out, and drawing them would imply a spatial arrangement the data does not
contain here. Node size is total outgoing synapse count; edge width is total synapse count
between the two types. The ON path is drawn in one colour and the OFF path in another.


In [ ]:
import networkx as nx

on_types = [MOTION_PATHWAYS["ON"]["input"], MOTION_PATHWAYS["ON"]["relay"], *MOTION_PATHWAYS["ON"]["outputs"]]
off_types = [MOTION_PATHWAYS["OFF"]["input"], *MOTION_PATHWAYS["OFF"]["relay"], *MOTION_PATHWAYS["OFF"]["outputs"]]
on_set, off_set = set(on_types), set(off_types)

graph = nx.DiGraph()
for node in list(MOTION_CELL_TYPES):
    graph.add_node(node)
max_syn = int(np.nanmax(matrix.to_numpy())) or 1
for src in matrix.index:
    for dst in matrix.columns:
        value = int(matrix.loc[src, dst])
        if value > 0 and src != dst:
            graph.add_edge(src, dst, weight=value)

positions = nx.spring_layout(graph, seed=0, k=0.7)
node_colors = [
    "#2ec4b6" if n in on_set else "#ef6f6c" if n in off_set else "#8892b0" for n in graph.nodes
]
node_sizes = [700 + 2600 * (float(matrix.loc[n, n].sum()) / max_syn) for n in graph.nodes]
edge_widths = [0.4 + 5.0 * (d["weight"] / max_syn) for _, _, d in graph.edges(data=True)]
edge_colors = [
    "#2ec4b6" if u in on_set else "#ef6f6c" if u in off_set else "#8892b0" for u, v in graph.edges
]

figure, axes = plt.subplots(figsize=(12, 9))
nx.draw_networkx_edges(graph, positions, ax=axes, width=edge_widths, edge_color=edge_colors, alpha=0.45)
nx.draw_networkx_nodes(graph, positions, ax=axes, node_color=node_colors, node_size=node_sizes, edgecolors="#0b1020")
nx.draw_networkx_labels(graph, positions, ax=axes, font_size=11, font_color="#0b1020", font_weight="bold")
axes.set_title(
    "MCNS motion candidates: cell-type connectivity (total integer synapse count)\n"
    "teal = ON channel (L1/Mi1/T4a-d)   coral = OFF channel (L2/Tm1,Tm2/T5a-d)   "
    "node size = within-type synapses",
    fontsize=10,
)
axes.set_axis_off()
figure.tight_layout()
plt.show()


## 10. Validation

Every required check, with its value. A failure here means the extraction is not
trustworthy and should not be used.


In [ ]:
validation = pd.DataFrame(report["validation"]["checks"])
validation["passed"] = validation["passed"].map({True: "PASS", False: "FAIL"})
print(f"ALL CHECKS PASSED: {report['validation']['all_passed']}")
print(f"self-loops retained (not removed): {report['validation']['self_loops']}")
validation[["passed", "check", "detail", "value"]]


In [ ]:
# Cell-type labels requested but absent from MCNS. Nothing is substituted.
missing = report["cell_type_labels"]["missing"]
if missing:
    print("LABELS NOT FOUND (not substituted):")
    for entry in missing:
        print(f"  {entry['label']}: near-miss strings {entry['near_miss_not_applied']} (NOT applied)")
else:
    print("all 13 requested cell types were found verbatim in the MCNS type field")


## 11. Save artifacts

Written to `outputs/mcns_motion/`, which is **git-ignored**. Nothing here is committed:
the raw data is not committed, and neither are derived graphs or summaries.


In [ ]:
written = write_report(report, OUTPUT_DIR / "mcns_motion_candidate_summary.json", OUTPUT_DIR)
for key, value in written.items():
    print(f"{key:<12} {value}")

print()
print("Confirm these paths are ignored by git:")
print("  git check-ignore -v outputs/mcns_motion/mcns_motion_candidate_summary.json")


## 12. What to record from this run

Copy these into your lab notes alongside the JSON artifact:

| Quantity | Value |
| --- | --- |
| candidate bodies | `report['populations']['candidate_bodies']` |
| edges retained | `report['extraction']['edges_retained']` |
| rows scanned | `report['extraction']['rows_scanned']` |
| record batches | `report['extraction']['record_batches_processed']` of `record_batches_total` |
| wall time | `report['total_seconds']` |
| retained bytes vs dense | `report['memory']` |
| all validations passed | `report['validation']['all_passed']` |
| is this a subset | `report['subset']['is_subset']` |

## 13. Honest reading of the result

What this extraction **is**: a measurement of how many synapses connect the candidate
cell populations, taken directly from the MaleCNS bulk files.

What it is **not**:

* **Not a circuit.** These are candidate populations, not a validated pathway. The ON and
  OFF chains are the standard literature framing; MCNS contains many other connections
  between these types, including strong feedback.
* **Not simulation.** No membrane potential has been computed.
* **Not weights.** `weight` is a synapse count. Converting a count into a synaptic
  conductance or efficacy is a modelling decision that has not been made and must not be
  made silently.
* **Not the visual system.** The candidate set is 13 cell types. Hoeller et al. 2026
  describe roughly 700 cell types in the right optic lobe alone, and this dataset has no
  functional annotation beyond the curated type labels.

Directionality deserves particular attention. The classic feedforward chain is present,
but the reverse connections in the matrix above are often *larger* by synapse count. A
strictly feedforward model would discard that, and the result would be a weaker claim than
the data supports.

**Next step is a decision, not more code:** which of these populations becomes the model,
at what spatial scale, and how synapse counts become weights. See
`docs/VISUAL_CIRCUIT_CANDIDATES.md` for the measured node budgets.
